## Quick tour

Get up and running with 🤗 Transformers! Whether you’re a developer or an everyday user, this quick tour will help you get started and show you how to use the pipeline() for inference, load a pretrained model and preprocessor with an AutoClass, and quickly train a model with PyTorch or TensorFlow. If you’re a beginner, we recommend checking out our tutorials or course next for more in-depth explanations of the concepts introduced here.

In [ ]:
#!pip install transformers datasets evaluate accelerate
#!pip install tf-keras

Start by creating an instance of pipeline() and specifying a task you want to use it for. In this guide, you’ll use the pipeline() for sentiment analysis as an example:

In [1]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use mps:0


The pipeline() downloads and caches a default pretrained model and tokenizer for sentiment analysis. Now you can use the classifier on your target text:

In [2]:
classifier("We are very happy to show you the 🤗 Transformers library.")

[{'label': 'POSITIVE', 'score': 0.9997795224189758}]

If you have more than one input, pass your inputs as a list to the pipeline() to return a list of dictionaries:

In [3]:
results = classifier(["We are very happy to show you the 🤗 Transformers library.", "We hope you don't hate it."])
for result in results:
    print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

label: POSITIVE, with score: 0.9998
label: NEGATIVE, with score: 0.5309


The pipeline() can also iterate over an entire dataset for any task you like. For this example, let’s choose automatic speech recognition as our task:

In [4]:
import torch
from transformers import pipeline

speech_recognizer = pipeline("automatic-speech-recognition", model="facebook/wav2vec2-base-960h")

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

Device set to use mps:0


Load an audio dataset (see the 🤗 Datasets Quick Start for more details) you’d like to iterate over. For example, load the MInDS-14 dataset:

In [ ]:
#!pip install soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 408.8 kB/s eta 0:00:00a 0:00:01


In [10]:
!pip install librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 573.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.2/26.2 MB 592.6 kB/s eta 0:00:0000:0100:02


In [11]:
from datasets import load_dataset, Audio

dataset = load_dataset("PolyAI/minds14", name="en-US", split="train")

You need to make sure the sampling rate of the dataset matches the sampling rate facebook/wav2vec2-base-960h was trained on:

In [12]:
dataset = dataset.cast_column("audio", Audio(sampling_rate=speech_recognizer.feature_extractor.sampling_rate))

The audio files are automatically loaded and resampled when calling the "audio" column. Extract the raw waveform arrays from the first 4 samples and pass it as a list to the pipeline:

In [17]:
from datasets import load_dataset, Audio
from transformers import pipeline

# Cargar el dataset
dataset = load_dataset("PolyAI/minds14", name="en-US", split="train")

# Configurar el pipeline de reconocimiento de voz
speech_recognizer = pipeline("automatic-speech-recognition", model="openai/whisper-small")

# Procesar los primeros 4 audios del dataset
audio_data = dataset[:4]["audio"]
result = speech_recognizer(audio_data)

# Imprimir los resultados
print([d["text"] for d in result])

model.safetensors:   7%|6         | 62.9M/967M [00:00<?, ?B/s]

Error while downloading from https://cdn-lfs.hf.co/repos/ca/81/ca8175f7fe33afb926eb59ad7c1664aaff834559c34d93520c9d26aff4b1e408/1d7734884874f1a1513ed9aa760a4f8e97aaa02fd6d93a3a85d27b2ae9ca596b?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1742829121&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MjgyOTEyMX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy9jYS84MS9jYTgxNzVmN2ZlMzNhZmI5MjZlYjU5YWQ3YzE2NjRhYWZmODM0NTU5YzM0ZDkzNTIwYzlkMjZhZmY0YjFlNDA4LzFkNzczNDg4NDg3NGYxYTE1MTNlZDlhYTc2MGE0ZjhlOTdhYWEwMmZkNmQ5M2EzYTg1ZDI3YjJhZTljYTU5NmI%7EcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIn1dfQ__&Signature=nYxHS8Bm5jNfrrlJYDTWhebHLj5sHy2GXXQ1EcYEL5M879lbFWyjBp-rgDnPLBu%7EGnzQbqIL8sOBvwvm-nhEKE0BiteWzgkdte8Lx0r2IpI65EG4wOzsGfAscpXh0JOhO5Wzf%7EOZAThBNga-yJsVr-HuQcERDKTiErP6zUXL23gOL9dkA5CdWvaV3fmYdx9FdQPej2V%7E5eItUQDGS3EI7696rkQ5yfPFjlPmHVwcK106dmM3LhN46PJnnacGcyTg964I4

model.safetensors:   8%|7         | 73.4M/967M [00:00<?, ?B/s]

Error while downloading from https://cdn-lfs.hf.co/repos/ca/81/ca8175f7fe33afb926eb59ad7c1664aaff834559c34d93520c9d26aff4b1e408/1d7734884874f1a1513ed9aa760a4f8e97aaa02fd6d93a3a85d27b2ae9ca596b?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1742829121&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MjgyOTEyMX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy9jYS84MS9jYTgxNzVmN2ZlMzNhZmI5MjZlYjU5YWQ3YzE2NjRhYWZmODM0NTU5YzM0ZDkzNTIwYzlkMjZhZmY0YjFlNDA4LzFkNzczNDg4NDg3NGYxYTE1MTNlZDlhYTc2MGE0ZjhlOTdhYWEwMmZkNmQ5M2EzYTg1ZDI3YjJhZTljYTU5NmI%7EcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIn1dfQ__&Signature=nYxHS8Bm5jNfrrlJYDTWhebHLj5sHy2GXXQ1EcYEL5M879lbFWyjBp-rgDnPLBu%7EGnzQbqIL8sOBvwvm-nhEKE0BiteWzgkdte8Lx0r2IpI65EG4wOzsGfAscpXh0JOhO5Wzf%7EOZAThBNga-yJsVr-HuQcERDKTiErP6zUXL23gOL9dkA5CdWvaV3fmYdx9FdQPej2V%7E5eItUQDGS3EI7696rkQ5yfPFjlPmHVwcK106dmM3LhN46PJnnacGcyTg964I4

pytorch_model.bin:   0%|          | 0.00/967M [00:00<?, ?B/s]

...

model.safetensors:  12%|#1        | 115M/967M [00:00<?, ?B/s]

Error while downloading from https://cdn-lfs.hf.co/repos/ca/81/ca8175f7fe33afb926eb59ad7c1664aaff834559c34d93520c9d26aff4b1e408/1d7734884874f1a1513ed9aa760a4f8e97aaa02fd6d93a3a85d27b2ae9ca596b?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1742828081&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MjgyODA4MX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy9jYS84MS9jYTgxNzVmN2ZlMzNhZmI5MjZlYjU5YWQ3YzE2NjRhYWZmODM0NTU5YzM0ZDkzNTIwYzlkMjZhZmY0YjFlNDA4LzFkNzczNDg4NDg3NGYxYTE1MTNlZDlhYTc2MGE0ZjhlOTdhYWEwMmZkNmQ5M2EzYTg1ZDI3YjJhZTljYTU5NmI%7EcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIn1dfQ__&Signature=SFxURLvf0BTsvD4msi%7EpxJQcBKgEf-XNQwVBQPEWY7%7E8EckF60IpEa3rbC42NTJWL0lfnYV0lD7bUHvSt3pUgFhv-dMdNmSZ%7EB3hgG59GZDrfXMAajWmb-epykGIvFI07wvn5hhCGGwWSMY6Tp8WnixCJ5EFKYbVA67MkQpUy6NUk0v%7ETIUBqvMquxaoGr5Ofk2vmAwZhy6FEXXjU5C92umrOTS21W2oMgk66nOp6LHpnYEkJclHndXAM9uJAD1kfVD

model.safetensors:  37%|###6      | 357M/967M [00:00<?, ?B/s]

Error while downloading from https://cdn-lfs.hf.co/repos/ca/81/ca8175f7fe33afb926eb59ad7c1664aaff834559c34d93520c9d26aff4b1e408/1d7734884874f1a1513ed9aa760a4f8e97aaa02fd6d93a3a85d27b2ae9ca596b?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1742828081&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MjgyODA4MX19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy9jYS84MS9jYTgxNzVmN2ZlMzNhZmI5MjZlYjU5YWQ3YzE2NjRhYWZmODM0NTU5YzM0ZDkzNTIwYzlkMjZhZmY0YjFlNDA4LzFkNzczNDg4NDg3NGYxYTE1MTNlZDlhYTc2MGE0ZjhlOTdhYWEwMmZkNmQ5M2EzYTg1ZDI3YjJhZTljYTU5NmI%7EcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qIn1dfQ__&Signature=SFxURLvf0BTsvD4msi%7EpxJQcBKgEf-XNQwVBQPEWY7%7E8EckF60IpEa3rbC42NTJWL0lfnYV0lD7bUHvSt3pUgFhv-dMdNmSZ%7EB3hgG59GZDrfXMAajWmb-epykGIvFI07wvn5hhCGGwWSMY6Tp8WnixCJ5EFKYbVA67MkQpUy6NUk0v%7ETIUBqvMquxaoGr5Ofk2vmAwZhy6FEXXjU5C92umrOTS21W2oMgk66nOp6LHpnYEkJclHndXAM9uJAD1kfVD

ValueError: Could not load model openai/whisper-small with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCTC'>, <class 'transformers.models.auto.modeling_auto.AutoModelForSpeechSeq2Seq'>, <class 'transformers.models.whisper.modeling_whisper.WhisperForConditionalGeneration'>, <class 'transformers.models.whisper.modeling_tf_whisper.TFWhisperForConditionalGeneration'>). See the original errors:

while loading with AutoModelForCTC, an error is thrown:
Traceback (most recent call last):
  File "/Users/bernardoquindimil/miniconda3/lib/python3.11/site-packages/transformers/pipelines/base.py", line 291, in infer_framework_load_model
    model = model_class.from_pretrained(model, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/bernardoquindimil/miniconda3/lib/python3.11/site-packages/transformers/models/auto/auto_factory.py", line 576, in from_pretrained
    raise ValueError(
ValueError: Unrecognized configuration cla

In [13]:
result = speech_recognizer(dataset[:4]["audio"])
print([d["text"] for d in result])

NotImplementedError: Output channels > 65536 not supported at the MPS device. 

For larger datasets where the inputs are big (like in speech or vision), you’ll want to pass a generator instead of a list to load all the inputs in memory. Take a look at the pipeline API reference for more information.